# P16 — Taleb Barbell Strategy Analysis

Systematic tail-risk hedging via deep OTM put options on SPY.  
Based on Nassim Taleb's barbell: safe base + convex tail hedge.

**Sections:** Setup → Data → Features → BS Primer → Simulation → Optimization → Multi-Strike → Regimes → Sensitivity → Export

In [ ]:
# Section 0 — Setup
import os
import sys
import warnings
from pathlib import Path

# Locate notebook directory so relative imports work regardless of launch CWD
try:
    _nb_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]  # VS Code
except NameError:
    _nb_dir = Path(os.path.abspath(""))

os.chdir(_nb_dir)

# Two separate paths to avoid the src/ naming collision:
#   _project_root_path  → project root, so that src.data.*, src.notification.* etc. resolve correctly
#   _pipeline_src       → pipeline's own src/, so data_loader/features/etc. are importable without prefix
_project_root_path = str(_nb_dir.parents[3])  # p16_taleb -> pipeline -> ml -> src -> project root
_pipeline_src = str(_nb_dir / "src")
for _p in [_pipeline_src, _project_root_path]:
    if _p not in sys.path:
        sys.path.insert(0, _p)
# Ensure project root is first so src.* resolves to the project package, not the pipeline src dir
if sys.path[0] != _project_root_path:
    sys.path.remove(_project_root_path)
    sys.path.insert(0, _project_root_path)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from IPython.display import HTML, display

warnings.filterwarnings("ignore", category=FutureWarning)

from data_loader import _project_root, load_all
from features import build_features
from optimizer import optimize_strikes
from pricing import bs_greeks, bs_put_price, skew_adjusted_sigma
from simulator import simulate_barbell
from visualizer import (
    chart_cumulative_pnl,
    chart_gdelt_tone,
    chart_heatmap,
    chart_pareto,
    chart_payoff_distribution,
    chart_premium_bleed,
    chart_sp500_drawdown,
    chart_vix_vs_cost,
    save_chart,
)

config = yaml.safe_load(open("config.yaml"))

CHARTS_DIR = _project_root / config["output"]["charts_path"]
RESULTS_DIR = _project_root / config["output"]["results_path"]
REPORT_PATH = _project_root / config["output"]["report_path"]
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {_project_root}")
print(f"Charts dir   : {CHARTS_DIR}")
print(f"Period       : {config['data']['start_date']} → {config['data']['end_date']}")

## Section 1 — Data Loading & Validation
Reads from the shared P15 `DATA_CACHE_DIR` (SPY OHLCV, ^VIX, DGS3MO, GDELT events).  
Merges on SPY trading days and caches to `results/p16_taleb/master_daily.csv.gz`.

In [2]:
master = load_all(config)

print(f"Shape      : {master.shape}")
print(f"Date range : {master.index.min().date()} → {master.index.max().date()}")
print(f"Columns    : {list(master.columns)}")
print()

missing = master.isna().sum()
missing = missing[missing > 0]
if len(missing):
    print("Missing values:")
    print(missing.to_string())
else:
    print("No missing values in core columns.")

master.head(3)

NameError: name 'load_all' is not defined

In [ ]:
# Gap validation: no run of missing business days > 5
bdays = pd.bdate_range(config["data"]["start_date"], config["data"]["end_date"])
missing_days = bdays.difference(master.index)
pct_missing = len(missing_days) / len(bdays) * 100

print(f"Expected business days : {len(bdays)}")
print(f"Master rows            : {len(master)}")
print(f"Missing business days  : {len(missing_days)} ({pct_missing:.2f}%)")

if len(missing_days) > 0:
    # Find consecutive runs
    diffs = pd.Series(missing_days).diff().dt.days.fillna(1)
    run_ids = (diffs > 3).cumsum()
    runs = pd.Series(missing_days).groupby(run_ids).agg(["first", "last", "count"])
    long_gaps = runs[runs["count"] >= 5]
    if len(long_gaps):
        print(f"WARNING: {len(long_gaps)} gap(s) longer than 5 business days:")
        print(long_gaps.to_string())
    else:
        print("All gaps < 5 business days. ✓")

master.tail(3)

## Section 2 — Feature Engineering
Computes returns, drawdown, VIX regimes, stress flags, and GDELT tone features.

In [ ]:
features_df = build_features(master)

new_cols = [c for c in features_df.columns if c not in master.columns]
print(f"Added feature columns: {new_cols}")
print()
print("VIX regime distribution:")
print(features_df["vix_regime"].astype(str).value_counts().sort_index().to_string())
print()
print(f"Stress flag rate : {features_df['stress_flag'].mean():.1%} of trading days")
print(
    f"GDELT available  : {features_df['gdelt_available'].sum() if 'gdelt_available' in features_df.columns else 0} days"
)

In [ ]:
fig1 = chart_sp500_drawdown(features_df)
fig1.show()
save_chart(fig1, "chart1_sp500_drawdown", CHARTS_DIR)

In [ ]:
# Major drawdown events table: local minima below -10%
dd = features_df[features_df["drawdown"] < -0.10][["close", "drawdown", "vix"]].copy()
dd["drawdown_pct"] = dd["drawdown"] * 100

# Group into distinct episodes (>30-day gap between rows = new episode)
gaps = dd.index.to_series().diff() > pd.Timedelta("30D")
episode_id = gaps.cumsum()
episodes = dd.groupby(episode_id).apply(lambda g: g.loc[g["drawdown"].idxmin()])

episodes_display = episodes[["close", "drawdown_pct", "vix"]].copy()
episodes_display.columns = ["S&P Price", "Drawdown %", "VIX at Trough"]
episodes_display.index = episodes_display.index.map(lambda x: str(x.date()) if hasattr(x, "date") else str(x))
print("Major Drawdown Episodes (> 10%):")
display(episodes_display.round(2))

## Section 3 — Black-Scholes Primer
Interactive exploration of put option pricing. Understand how S, K, T, r, σ affect price and Greeks.

In [ ]:
import ipywidgets as widgets


def _bs_panel(S, K, T_days, r_pct, sigma_pct):
    T = T_days / 365.0
    r = r_pct / 100.0
    sigma = sigma_pct / 100.0
    price = bs_put_price(S, K, T, r, sigma)
    g = bs_greeks(S, K, T, r, sigma)
    otm = (1.0 - K / S) * 100.0
    breakeven = price / S * 100.0
    skew_sigma = skew_adjusted_sigma(sigma_pct, max(0.0, otm / 100.0)) * 100.0
    skew_price = bs_put_price(S, K, T, r, skew_sigma / 100.0)

    html = f"""
    <div style='font-family:monospace; background:#1e1e2e; color:#cdd6f4; padding:16px;
                border-radius:8px; display:grid; grid-template-columns:1fr 1fr; gap:8px 24px;'>
      <div><b style='color:#89b4fa'>OTM %</b><br>{otm:.2f}%</div>
      <div><b style='color:#89b4fa'>Flat-vol price</b><br>${price:.4f} ({price / S * 100:.3f}% of S)</div>
      <div><b style='color:#89b4fa'>Skew-adj σ</b><br>{skew_sigma:.2f}%</div>
      <div><b style='color:#89b4fa'>Skew-adj price</b><br>${skew_price:.4f} ({skew_price / S * 100:.3f}% of S)</div>
      <div style='color:#f38ba8'><b>Delta</b><br>{g["delta"]:.4f}</div>
      <div><b>Gamma</b><br>{g["gamma"]:.6f}</div>
      <div style='color:#f38ba8'><b>Theta ($/day)</b><br>{g["theta"]:.4f}</div>
      <div style='color:#a6e3a1'><b>Vega ($/1% vol)</b><br>{g["vega"]:.4f}</div>
      <div><b>Rho ($/1% rate)</b><br>{g["rho"]:.4f}</div>
      <div style='color:#fab387'><b>Breakeven drop</b><br>-{breakeven:.3f}%</div>
    </div>
    """
    display(HTML(html))


S_ref = float(features_df["close"].iloc[-1])
widgets.interact(
    _bs_panel,
    S=widgets.FloatSlider(value=round(S_ref, -2), min=100, max=8000, step=50, description="S (spot)"),
    K=widgets.FloatSlider(value=round(S_ref * 0.85, -2), min=100, max=8000, step=50, description="K (strike)"),
    T_days=widgets.IntSlider(value=90, min=7, max=365, step=7, description="T (days)"),
    r_pct=widgets.FloatSlider(value=4.0, min=0.0, max=10.0, step=0.25, description="r (%)"),
    sigma_pct=widgets.FloatSlider(value=20.0, min=5.0, max=80.0, step=1.0, description="σ (%)"),
);

In [ ]:
# BS put price surface over (moneyness, tenor) grid — σ=20%, r=4%
S_ref = 4000.0
T_axis = np.linspace(7 / 365, 365 / 365, 50)
m_axis = np.linspace(0.60, 1.00, 50)

Z = np.array([[bs_put_price(S_ref, m * S_ref, t, 0.04, 0.20) / S_ref * 100 for m in m_axis] for t in T_axis])

fig_surf = go.Figure(
    data=go.Surface(
        x=m_axis,
        y=(T_axis * 365).astype(int),
        z=Z,
        colorscale="Viridis",
        colorbar=dict(title="Put % of S"),
    )
)
fig_surf.update_layout(
    template="plotly_dark",
    title="Black-Scholes Put Price Surface (σ=20%, r=4%)",
    scene=dict(
        xaxis_title="K/S (moneyness)",
        yaxis_title="T (days)",
        zaxis_title="Put Price (% of S)",
    ),
    height=600,
)
fig_surf.show()

## Section 4 — Single Strategy Simulation
Baseline run using config defaults: 15% OTM, 90-day tenor, rebalance every 21 trading days.

In [ ]:
sim = simulate_barbell(
    features_df,
    moneyness=config["strategy"]["moneyness"],
    T_days=config["strategy"]["T_days"],
    rebalance_days=config["strategy"]["rebalance_days"],
    put_budget_pct=config["strategy"]["put_budget_pct"],
    initial_capital=config["strategy"]["initial_capital"],
    skew_slope=config["pricing"]["skew_slope"],
)

total_cost = sim["budget_spent"].sum()
total_payoff = sim["payoff"].sum()
net_pnl = sim["pnl"].sum()
win_rate = (sim["payoff"] > 0).mean()
annual_cost_pct = config["strategy"]["put_budget_pct"] * 100 * 252 / config["strategy"]["rebalance_days"]

print(
    f"Strategy    : {config['strategy']['moneyness']:.0%} moneyness, "
    f"{config['strategy']['T_days']}d tenor, "
    f"rebalance every {config['strategy']['rebalance_days']} trading days"
)
print(f"Periods     : {len(sim)}")
print(f"Annual cost : ~{annual_cost_pct:.1f}% of capital")
print(f"Total cost  : ${total_cost:,.0f}")
print(f"Total payoff: ${total_payoff:,.0f}")
print(f"Net P&L     : ${net_pnl:,.0f}  ({net_pnl / total_cost * 100:.1f}% of premium spent)")
print(f"Win rate    : {win_rate:.1%}")
print(f"Avg premium : {sim['put_price_pct_of_S'].mean():.3f}% of S")

In [ ]:
# Top 10 payoff events
top10 = (
    sim[sim["payoff"] > 0]
    .nlargest(10, "payoff")[["S", "K", "S_at_expiry", "drawdown_at_expiry", "vix", "payoff", "pnl"]]
    .copy()
)
top10["drawdown_at_expiry"] = (top10["drawdown_at_expiry"] * 100).round(1)
top10.columns = ["S (buy)", "Strike K", "S (expiry)", "DD at expiry %", "VIX", "Payoff $", "Net P&L $"]
print("Top 10 payoff events:")
display(top10.round(2))

In [ ]:
fig3 = chart_cumulative_pnl({"15% OTM, 90d": sim}, df=features_df)
fig3.show()
save_chart(fig3, "chart3_cumulative_pnl", CHARTS_DIR)

fig5 = chart_premium_bleed(sim)
fig5.show()
save_chart(fig5, "chart5_premium_bleed", CHARTS_DIR)

fig4 = chart_payoff_distribution(sim)
fig4.show()
save_chart(fig4, "chart4_payoff_distribution", CHARTS_DIR)

fig6 = chart_vix_vs_cost(sim)
fig6.show()
save_chart(fig6, "chart6_vix_vs_cost", CHARTS_DIR)

## Section 5 — Strike Optimization
Grid search over (moneyness × T_days × rebalance_days) using `ProcessPoolExecutor`. ~60 seconds.

In [ ]:
print("Running optimizer...  (uses all CPU cores, ~60 seconds)")
opt_df = optimize_strikes(
    features_df,
    moneyness_grid=config["optimization"]["moneyness_grid"],
    T_days_grid=config["optimization"]["T_days_grid"],
    rebalance_days_grid=config["optimization"]["rebalance_days_grid"],
    budget_pct=config["strategy"]["put_budget_pct"],
    initial_capital=config["strategy"]["initial_capital"],
    skew_slope=config["pricing"]["skew_slope"],
)

opt_path = RESULTS_DIR / "optimization_results.csv.gz"
opt_df.to_csv(opt_path, compression="gzip")
print(f"Tested {len(opt_df)} combinations → saved {opt_path.name}")

In [ ]:
# Top 10 by net ROI
cols = [
    "strike_otm_pct",
    "T_days",
    "net_roi_pct",
    "win_rate_pct",
    "crisis_capture_rate",
    "sharpe_analog",
    "total_pnl",
    "n_crisis_periods",
]
display(opt_df[cols].head(10).round(2))

print()
print("Best strategy by each objective:")
for obj in ["net_roi_pct", "win_rate_pct", "crisis_capture_rate", "payoff_to_cost_ratio"]:
    best = opt_df.loc[opt_df[obj].idxmax()]
    print(f"  {obj:30s}: {best['strike_otm_pct']:.0f}% OTM, {best['T_days']:.0f}d ({obj} = {best[obj]:.2f})")

In [ ]:
fig2 = chart_heatmap(opt_df)
fig2.show()
save_chart(fig2, "chart2_heatmap", CHARTS_DIR)

fig8 = chart_pareto(opt_df)
fig8.show()
save_chart(fig8, "chart8_pareto", CHARTS_DIR)

## Section 6 — Multi-Strike Comparison
Run 5 representative strikes (selected from optimizer Pareto frontier) and compare side-by-side.

In [ ]:
# Pick 5 distinct OTM levels evenly spaced across the grid (T=90d)
pool = opt_df[opt_df["T_days"] == 90].sort_values("strike_otm_pct")
n = len(pool)
selected_rows = pool.iloc[[0, n // 4, n // 2, 3 * n // 4, n - 1]]

multi_sims: dict = {}
for _, row in selected_rows.iterrows():
    label = f"{row['strike_otm_pct']:.0f}% OTM, {row['T_days']:.0f}d"
    multi_sims[label] = simulate_barbell(
        features_df,
        moneyness=float(row["moneyness"]),
        T_days=int(row["T_days"]),
        rebalance_days=config["strategy"]["rebalance_days"],
        put_budget_pct=config["strategy"]["put_budget_pct"],
        initial_capital=config["strategy"]["initial_capital"],
        skew_slope=config["pricing"]["skew_slope"],
    )
    s = multi_sims[label]
    roi = s["pnl"].sum() / s["budget_spent"].sum() * 100 if not s.empty else 0
    print(f"  {label}: ROI={roi:.1f}%,  win={((s['payoff'] > 0).mean()):.1%}")

In [ ]:
# Comparison table
rows = []
for label, s in multi_sims.items():
    if s.empty:
        continue
    rows.append(
        {
            "Strategy": label,
            "Total Cost": f"${s['budget_spent'].sum():,.0f}",
            "Total Payoff": f"${s['payoff'].sum():,.0f}",
            "Net P&L": f"${s['pnl'].sum():,.0f}",
            "Net ROI": f"{s['pnl'].sum() / s['budget_spent'].sum() * 100:.1f}%",
            "Win Rate": f"{(s['payoff'] > 0).mean():.1%}",
            "Max Payoff": f"${s['payoff'].max():,.0f}",
        }
    )
display(pd.DataFrame(rows).set_index("Strategy"))

# Crisis highlights: best payer per crisis window
print()
print("Largest payoff per crisis window:")
crisis_windows = [
    ("US Downgrade 2011", "2011-06", "2011-11"),
    ("China Crash 2015", "2015-06", "2015-11"),
    ("Q4 Selloff 2018", "2018-10", "2019-01"),
    ("COVID-19 2020", "2020-01", "2020-06"),
    ("Fed Hike 2022", "2022-03", "2022-09"),
]
for name, start, end in crisis_windows:
    best_label, best_pay = "", 0.0
    for label, s in multi_sims.items():
        window_pay = s.loc[start:end, "payoff"].sum() if not s.empty else 0.0
        if window_pay > best_pay:
            best_pay, best_label = window_pay, label
    winner = f"{best_label}: ${best_pay:,.0f}" if best_pay > 0 else "No payoff"
    print(f"  {name:25s} → {winner}")

In [ ]:
fig3m = chart_cumulative_pnl(multi_sims, df=features_df)
fig3m.update_layout(title="Cumulative P&L — Multi-Strike Comparison")
fig3m.show()
save_chart(fig3m, "chart3_multistrike", CHARTS_DIR)

## Section 7 — Regime Analysis
Split simulation output by VIX regime.  
Key insight: puts are most expensive exactly when crises are most likely.

In [ ]:
# Join regime labels onto simulation output
sim_r = sim.join(features_df[["vix_regime", "stress_flag"]], how="left")
sim_r["vix_regime_str"] = sim_r["vix_regime"].astype(str)

regime_stats = (
    sim_r.groupby("vix_regime_str")
    .agg(
        periods=("payoff", "count"),
        avg_put_pct=("put_price_pct_of_S", "mean"),
        win_rate=("payoff", lambda x: (x > 0).mean()),
        avg_payoff=("payoff", "mean"),
        total_pnl=("pnl", "sum"),
    )
    .reindex(["low", "normal", "elevated", "high", "extreme"])
    .dropna(how="all")
)

regime_stats["win_rate"] = regime_stats["win_rate"].map("{:.1%}".format)
regime_stats.columns = ["Periods", "Avg put % of S", "Win Rate", "Avg Payoff $", "Total P&L $"]
print("Strategy performance by VIX regime:")
display(regime_stats.round(4))

print()
low_mask = sim_r["vix_regime_str"] == "low"
high_mask = sim_r["vix_regime_str"].isin(["high", "extreme"])
if low_mask.any() and high_mask.any():
    low_cost = sim_r.loc[low_mask, "put_price_pct_of_S"].mean()
    low_wr = (sim_r.loc[low_mask, "payoff"] > 0).mean()
    hi_cost = sim_r.loc[high_mask, "put_price_pct_of_S"].mean()
    hi_wr = (sim_r.loc[high_mask, "payoff"] > 0).mean()
    print("KEY INSIGHT:")
    print(f"  Low VIX  (<15): avg cost = {low_cost:.3f}% of S, win rate = {low_wr:.1%}")
    print(f"  High VIX (>25): avg cost = {hi_cost:.3f}% of S, win rate = {hi_wr:.1%}")
    print(
        f"  Puts are {hi_cost / low_cost:.1f}x more expensive in high-VIX regimes, but crises are {hi_wr / max(low_wr, 0.001):.1f}x more likely."
    )

In [ ]:
# Chart 7: GDELT tone (shows placeholder if GDELT data not available)
fig7 = chart_gdelt_tone(features_df)
fig7.show()
save_chart(fig7, "chart7_gdelt_tone", CHARTS_DIR)

## Section 8 — Sensitivity Analysis
Vary `put_budget_pct`, `T_days`, and `rebalance_days` independently to understand their impact.

In [ ]:
# Budget sensitivity
print("=== put_budget_pct sensitivity (T=90d, 15% OTM) ===")
budget_rows = []
for bp in [0.005, 0.010, 0.020, 0.030, 0.050]:
    s = simulate_barbell(
        features_df,
        moneyness=config["strategy"]["moneyness"],
        T_days=config["strategy"]["T_days"],
        rebalance_days=config["strategy"]["rebalance_days"],
        put_budget_pct=bp,
        initial_capital=config["strategy"]["initial_capital"],
    )
    if s.empty:
        continue
    annual = bp * 100 * 252 / config["strategy"]["rebalance_days"]
    budget_rows.append(
        {
            "Budget/period": f"{bp:.1%}",
            "Annual cost ~": f"{annual:.1f}%",
            "Total cost $": f"${s['budget_spent'].sum():,.0f}",
            "Total payoff $": f"${s['payoff'].sum():,.0f}",
            "Net P&L $": f"${s['pnl'].sum():,.0f}",
            "Net ROI": f"{s['pnl'].sum() / s['budget_spent'].sum() * 100:.1f}%",
        }
    )
display(pd.DataFrame(budget_rows).set_index("Budget/period"))

In [ ]:
# Tenor sensitivity
print("=== T_days sensitivity (15% OTM, 2% budget) ===")
T_rows = []
for T in [30, 60, 90, 120, 180]:
    s = simulate_barbell(
        features_df,
        moneyness=config["strategy"]["moneyness"],
        T_days=T,
        rebalance_days=config["strategy"]["rebalance_days"],
        put_budget_pct=config["strategy"]["put_budget_pct"],
        initial_capital=config["strategy"]["initial_capital"],
    )
    if s.empty:
        continue
    T_rows.append(
        {
            "T (days)": T,
            "Avg put cost %": f"{s['put_price_pct_of_S'].mean():.3f}%",
            "Win rate": f"{(s['payoff'] > 0).mean():.1%}",
            "Max payoff $": f"${s['payoff'].max():,.0f}",
            "Net P&L $": f"${s['pnl'].sum():,.0f}",
            "Net ROI": f"{s['pnl'].sum() / s['budget_spent'].sum() * 100:.1f}%",
        }
    )
display(pd.DataFrame(T_rows).set_index("T (days)"))
print("Short-dated puts are cheaper but may expire before the crash recovery.")

In [ ]:
# Rebalance frequency sensitivity
print("=== rebalance_days sensitivity (15% OTM, 90d tenor) ===")
rebal_rows = []
for r in [5, 10, 21, 42]:
    s = simulate_barbell(
        features_df,
        moneyness=config["strategy"]["moneyness"],
        T_days=config["strategy"]["T_days"],
        rebalance_days=r,
        put_budget_pct=config["strategy"]["put_budget_pct"],
        initial_capital=config["strategy"]["initial_capital"],
    )
    if s.empty:
        continue
    rebal_rows.append(
        {
            "Rebalance days": r,
            "Periods/year ~": f"{252 // r}",
            "Annual cost ~": f"{config['strategy']['put_budget_pct'] * 100 * (252 // r):.1f}%",
            "Win rate": f"{(s['payoff'] > 0).mean():.1%}",
            "Net P&L $": f"${s['pnl'].sum():,.0f}",
            "Net ROI": f"{s['pnl'].sum() / s['budget_spent'].sum() * 100:.1f}%",
        }
    )
display(pd.DataFrame(rebal_rows).set_index("Rebalance days"))
print("More frequent rebalancing = higher annual cost but more granular crash coverage.")

## Section 9 — Real Options Prices vs BS *(conditional)*
Only executes when `config.pricing.use_market_prices = true` and historical chains are present.

In [ ]:
if config["pricing"]["use_market_prices"]:
    # Load real options chains from DATA_CACHE_DIR (requires historical data)
    # Expected columns: date, S, K, T, bid, ask, iv_market
    chains_path = _project_root / "data" / "raw" / "options_chains"
    chain_files = sorted(chains_path.glob("*.csv.gz"))

    if not chain_files:
        print("No options chain files found. Populate data/raw/options_chains/ first.")
    else:
        sample_dates = chain_files[:5]  # first 5 files
        chains = pd.concat([pd.read_csv(f, compression="gzip") for f in sample_dates])

        # Compute BS theoretical price for each row
        chains["bs_price"] = chains.apply(
            lambda row: bs_put_price(row["S"], row["K"], row["T"], row.get("rate", 0.04), row.get("iv_market", 0.20)),
            axis=1,
        )
        chains["mid_price"] = (chains["bid"] + chains["ask"]) / 2
        chains["overpay_factor"] = chains["mid_price"] / chains["bs_price"].clip(lower=0.001)
        chains["spread_pct"] = (chains["ask"] - chains["bid"]) / chains["mid_price"] * 100

        print(f"Loaded {len(chains)} option contracts from {len(sample_dates)} dates.")
        print(
            f"Overpay factor (mid/BS): mean={chains['overpay_factor'].mean():.3f}, "
            f"median={chains['overpay_factor'].median():.3f}"
        )
        print(f"Bid/ask spread: mean={chains['spread_pct'].mean():.1f}% of mid")
        display(chains[["S", "K", "mid_price", "bs_price", "overpay_factor", "spread_pct"]].head(10).round(4))
else:
    print("Section 9 skipped: config.pricing.use_market_prices = false")
    print()
    print("To enable:")
    print("  1. Set use_market_prices: true in config.yaml")
    print("  2. Populate data/raw/options_chains/ with historical SPY option chain CSV.gz files")
    print("     Expected columns: date, S, K, T, bid, ask, iv_market")

## Section 10 — Summary & Recommendations
Auto-generated summary, COVID-19 crash case study, and full HTML report export.

In [ ]:
best_roi = opt_df.loc[opt_df["net_roi_pct"].idxmax()]
best_crisis = opt_df.dropna(subset=["crisis_capture_rate"]).loc[
    opt_df.dropna(subset=["crisis_capture_rate"])["crisis_capture_rate"].idxmax()
]
best_winrate = opt_df.loc[opt_df["win_rate_pct"].idxmax()]

print("=" * 62)
print("  P16 TALEB BARBELL — RESULTS SUMMARY")
print("=" * 62)
print(f"  Period    : {config['data']['start_date']} → {config['data']['end_date']}")
print(f"  Universe  : {config['data']['sp500_ticker']} (S&P 500 proxy)")
print(f"  Budget    : {config['strategy']['put_budget_pct']:.1%} of capital per period")
print()
print("  DEFAULT STRATEGY (15% OTM, 90-day puts, rebal every 21d):")
print(f"    Periods         : {len(sim)}")
print(f"    Total cost      : ${sim['budget_spent'].sum():,.0f}")
print(f"    Total payoff    : ${sim['payoff'].sum():,.0f}")
print(f"    Net P&L         : ${sim['pnl'].sum():,.0f}")
print(f"    Win rate        : {(sim['payoff'] > 0).mean():.1%}")
print()
print("  OPTIMAL STRATEGIES (from grid search):")
print(
    f"    Best ROI        : {best_roi['strike_otm_pct']:.0f}% OTM, {best_roi['T_days']:.0f}d  "
    f"(ROI={best_roi['net_roi_pct']:.1f}%, win={best_roi['win_rate_pct']:.1f}%)"
)
print(
    f"    Best crisis cap : {best_crisis['strike_otm_pct']:.0f}% OTM, {best_crisis['T_days']:.0f}d  "
    f"(capture={best_crisis['crisis_capture_rate']:.1f}%)"
)
print(
    f"    Best win rate   : {best_winrate['strike_otm_pct']:.0f}% OTM, {best_winrate['T_days']:.0f}d  "
    f"(win={best_winrate['win_rate_pct']:.1f}%)"
)
print("=" * 62)

In [ ]:
# COVID-19 2020 crash case study
print("COVID-19 CRASH CASE STUDY")
print("-" * 40)

try:
    jan2020 = features_df.loc["2020-01-02", "close"]
except KeyError:
    jan2020 = features_df.loc["2020-01"].iloc[0]["close"]

trough_idx = features_df.loc["2020-03", "close"].idxmin()
trough_price = features_df.loc[trough_idx, "close"]
crash_pct = (trough_price - jan2020) / jan2020 * 100

covid_cost = sim.loc["2020-01":"2020-03", "budget_spent"].sum()
covid_payoff = sim.loc["2020-01":"2020-06", "payoff"].sum()
covid_net = covid_payoff - covid_cost

initial = float(config["strategy"]["initial_capital"])
port_naked = initial * (1 + crash_pct / 100)
port_hedged = port_naked + covid_payoff - covid_cost

print(f"S&P 500 peak-to-trough : {crash_pct:.1f}% (trough: {trough_idx.date()})")
print(f"Put premiums paid      : ${covid_cost:,.0f} (Jan–Mar 2020)")
print(f"Payoffs received       : ${covid_payoff:,.0f}")
print(f"Net insurance value    : ${covid_net:,.0f}")
print()
print("$100k portfolio at trough:")
print(f"  Without puts : ${port_naked:,.0f}  (loss ${port_naked - initial:,.0f})")
print(f"  With puts    : ${port_hedged:,.0f}  (loss ${port_hedged - initial:,.0f})")
if port_naked < initial:
    protection_pct = (port_hedged - port_naked) / (initial - port_naked) * 100
    print(f"  Puts offset {protection_pct:.0f}% of the crash loss.")

In [ ]:
# Export all charts to a single HTML report
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

all_figs = [
    ("S&P 500 — Price, Drawdown & VIX", fig1),
    ("Strike Optimization Heatmap", fig2),
    ("Cumulative P&L — Default Strategy", fig3),
    ("P&L Distribution", fig4),
    ("Premium Bleed vs Payoff", fig5),
    ("VIX vs Option Cost", fig6),
    ("GDELT Tone vs Drawdown", fig7),
    ("Multi-Strike Comparison", fig3m),
    ("Pareto Frontier", fig8),
]

header = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>P16 Taleb Barbell Analysis</title>
<style>
body{{background:#0d0d1a;color:#e0e0e0;font-family:'Segoe UI',sans-serif;margin:24px;max-width:1400px}}
h1{{color:#00B4D8;border-bottom:2px solid #00B4D8;padding-bottom:8px}}
h2{{color:#90E0EF;margin-top:32px;border-bottom:1px solid #333;padding-bottom:4px}}
.meta{{color:#888;margin-bottom:24px}}
</style></head><body>
<h1>P16 — Taleb Barbell Strategy Analysis</h1>
<p class="meta">Period: {config["data"]["start_date"]} &rarr; {config["data"]["end_date"]} &nbsp;|
 Ticker: {config["data"]["sp500_ticker"]} &nbsp;|
 Budget: {config["strategy"]["put_budget_pct"]:.1%}/period &nbsp;|
 Strategy: {config["strategy"]["moneyness"]:.0%} moneyness, {config["strategy"]["T_days"]}d tenor</p>
"""

parts = [header]
for i, (title, fig) in enumerate(all_figs):
    include_js = "cdn" if i == 0 else False
    parts.append(f"<h2>{title}</h2>")
    parts.append(fig.to_html(full_html=False, include_plotlyjs=include_js))

parts.append("</body></html>")

REPORT_PATH.write_text("\n".join(parts), encoding="utf-8")
print(f"Report exported → {REPORT_PATH}")
print(f"Charts saved    → {CHARTS_DIR}")
print(f"Results saved   → {RESULTS_DIR}")